# Trading Data Exploratory Data Analysis

This notebook performs a comprehensive exploratory data analysis of our cryptocurrency trading data and ML model performance.

In [1]:
# Import required libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
import talib as ta
from datetime import datetime, timedelta
import warnings
warnings.filterwarnings('ignore')

# Set style for better visualizations
plt.style.use('default')  # Use default style first
sns.set_theme(style="whitegrid")  # Apply seaborn theme
plt.rcParams['figure.figsize'] = [12, 8]
plt.rcParams['figure.dpi'] = 100
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3

## 1. Load and Prepare Data

In [ ]:
def add_technical_indicators(df):
    """Add technical indicators using the ta library."""
    df = df.copy()
    
    # Initialize ta indicators
    import ta
    
    # Trend Indicators
    # Bollinger Bands
    bb = ta.volatility.BollingerBands(df['close'])
    df['BB_upper'] = bb.bollinger_hband()
    df['BB_middle'] = bb.bollinger_mavg()
    df['BB_lower'] = bb.bollinger_lband()
    
    # Moving Averages
    df['SMA_20'] = ta.trend.sma_indicator(df['close'], window=20)
    df['SMA_50'] = ta.trend.sma_indicator(df['close'], window=50)
    df['EMA_20'] = ta.trend.ema_indicator(df['close'], window=20)
    
    # MACD
    macd = ta.trend.MACD(df['close'])
    df['MACD'] = macd.macd()
    df['MACD_signal'] = macd.macd_signal()
    df['MACD_hist'] = macd.macd_diff()
    
    # Momentum Indicators
    # RSI
    df['RSI'] = ta.momentum.RSIIndicator(df['close']).rsi()
    
    # Stochastic RSI
    stoch_rsi = ta.momentum.StochRSIIndicator(df['close'])
    df['StochRSI_k'] = stoch_rsi.stochrsi_k()
    df['StochRSI_d'] = stoch_rsi.stochrsi_d()
    
    # Volatility Indicators
    # ATR
    df['ATR'] = ta.volatility.AverageTrueRange(df['high'], df['low'], df['close']).average_true_range()
    
    # Volume Indicators
    # Calculate Volume SMA manually
    df['Volume_SMA_20'] = df['volume'].rolling(window=20).mean()
    df['Volume_ratio'] = df['volume'] / df['Volume_SMA_20']
    
    # MFI (Money Flow Index)
    df['MFI'] = ta.volume.money_flow_index(df['high'], df['low'], df['close'], df['volume'])
    
    # Additional Indicators
    # ADX (Average Directional Index)
    adx = ta.trend.ADXIndicator(df['high'], df['low'], df['close'])
    df['ADX'] = adx.adx()
    df['ADX_pos'] = adx.adx_pos()
    df['ADX_neg'] = adx.adx_neg()
    
    # Commodity Channel Index
    df['CCI'] = ta.trend.cci(df['high'], df['low'], df['close'])
    
    # Williams %R
    df['Williams_R'] = ta.momentum.WilliamsRIndicator(df['high'], df['low'], df['close']).williams_r()
    
    # Returns and Volatility
    df['returns'] = df['close'].pct_change()
    df['log_returns'] = np.log(df['close']).diff()
    df['volatility'] = df['returns'].rolling(window=20).std() * np.sqrt(365 * 24 * 4)  # Annualized
    
    # Clean up any infinite values
    df.replace([np.inf, -np.inf], np.nan, inplace=True)
    
    return df
def load_data(symbol='BTCUSDT', min_periods=30):
    """
    Load and prepare trading data with technical indicators.
    
    Args:
        symbol (str): Trading pair symbol
        min_periods (int): Minimum number of periods to load for calculating indicators
    """
    from pathlib import Path
    
    # Find data files
    data_files = list(Path('historical_data').glob(f'{symbol}_data_*.csv'))
    
    if not data_files:
        raise FileNotFoundError(f"No data files found for {symbol}")
    
    # Load the most recent file
    latest_file = max(data_files, key=lambda x: x.stat().st_mtime)
    
    # Read CSV with proper parsing
    df = pd.read_csv(
        latest_file,
        parse_dates=['timestamp']
    )
    
    # Sort by timestamp and set as index
    df.sort_values('timestamp', inplace=True)
    df.set_index('timestamp', inplace=True)
    
    # Add technical indicators
    df = add_technical_indicators(df)
    
    # Drop NaN values from the beginning of the dataset
    df = df.dropna()
    
    print(f"\nLoaded data shape: {df.shape}")
    print(f"Date range: {df.index.min()} to {df.index.max()}")
    print("\nNew technical indicators added:")
    print(", ".join([col for col in df.columns if col not in ['open', 'high', 'low', 'close', 'volume', 
                    'close_time', 'quote_asset_volume', 'number_of_trades', 
                    'taker_buy_base_asset_volume', 'taker_buy_quote_asset_volume', 'ignore']]))
    
    return df

# Load data with technical indicators
df = load_data(min_periods=30)
display(df.head())

# Print some basic statistics to verify calculations
print("\nBasic statistics for key indicators:")
indicators = ['RSI', 'MACD', 'ATR', 'volatility', 'MFI', 'ADX', 'CCI', 'Williams_R']
print(df[indicators].describe())

## 2. Basic Price Analysis

In [ ]:
def plot_candlestick_volume(df):
    # Calculate Bollinger Bands first
    df = df.copy()
    
    # Create figure with secondary y-axis
    fig = make_subplots(rows=2, cols=1, shared_xaxes=True, 
                        vertical_spacing=0.03, subplot_titles=('Price', 'Volume'),
                        row_heights=[0.7, 0.3])

    # Add candlestick
    fig.add_trace(go.Candlestick(x=df.index,
                                open=df['open'],
                                high=df['high'],
                                low=df['low'],
                                close=df['close'],
                                name='OHLC'),
                  row=1, col=1)

    # Add Bollinger Bands
    fig.add_trace(go.Scatter(x=df.index, y=df['BB_upper'],
                            line=dict(color='rgba(255,0,0,0.3)'),
                            name='Bollinger Upper'), row=1, col=1)
    fig.add_trace(go.Scatter(x=df.index, y=df['BB_middle'],
                            line=dict(color='rgba(128,128,128,0.3)'),
                            name='Bollinger Middle'), row=1, col=1)
    fig.add_trace(go.Scatter(x=df.index, y=df['BB_lower'],
                            line=dict(color='rgba(0,255,0,0.3)'),
                            name='Bollinger Lower'), row=1, col=1)

    # Add volume bar chart
    colors = ['red' if row['close'] < row['open'] else 'green' for idx, row in df.iterrows()]
    fig.add_trace(go.Bar(x=df.index, 
                        y=df['volume'],
                        marker_color=colors,
                        name='Volume'),
                  row=2, col=1)
    
    # Add volume MA
    fig.add_trace(go.Scatter(x=df.index, 
                            y=df['volume'].rolling(20).mean(),
                            line=dict(color='yellow', width=2),
                            name='Volume MA(20)'), 
                  row=2, col=1)
    
    # Enhanced layout
    fig.update_layout(
        height=800,
        title='Price Action with Bollinger Bands and Volume',
        hovermode='x unified',
        showlegend=True,
        xaxis=dict(
            rangeselector=dict(
                buttons=list([
                    dict(count=1, label="1h", step="hour", stepmode="backward"),
                    dict(count=4, label="4h", step="hour", stepmode="backward"),
                    dict(count=1, label="1d", step="day", stepmode="backward"),
                    dict(step="all")
                ])
            )
        )
    )
    
    # Update y-axes labels
    fig.update_yaxes(title_text="Price", row=1, col=1)
    fig.update_yaxes(title_text="Volume", row=2, col=1)
    
    fig.show()

plot_candlestick_volume(df)

## 3. Technical Indicators Analysis

In [ ]:
def plot_technical_indicators(df):
    # Create 3x2 grid for better organization
    fig = make_subplots(rows=3, cols=2, specs=[[{}, {}], [{}, {}], [{"colspan": 2}, None]],
                        subplot_titles=('Price & MAs', 'Bollinger Bands', 
                                      'RSI', 'MACD', 'Volatility & ATR'))
    
    # Price & MAs with Bollinger Bands
    fig.add_trace(go.Candlestick(x=df.index, open=df['open'],
                               high=df['high'], low=df['low'], close=df['close']), row=1, col=1)

    fig.add_trace(go.Scatter(x=df.index, y=df['BB_upper'], 
                           line=dict(color='red'), name='BB Upper'), row=1, col=2)
    fig.add_trace(go.Scatter(x=df.index, y=df['BB_lower'],
                           line=dict(color='green'), name='BB Lower'), row=1, col=2)
    
    # RSI with histogram
    fig.add_trace(go.Bar(x=df.index, y=df['RSI']-50, 
                       marker_color=np.where(df['RSI']>50, 'green', 'red'),
                       name='RSI Strength'), row=2, col=1)
    
    # MACD histogram
    fig.add_trace(go.Bar(x=df.index, y=df['MACD_hist'],
                       marker_color=np.where(df['MACD_hist']>0, 'green', 'red'),
                       name='MACD Hist'), row=2, col=2)
    
    # Volatility & ATR
    fig.add_trace(go.Scatter(x=df.index, y=df['volatility'],
                           line=dict(color='purple'), name='Volatility'), row=3, col=1)
    fig.add_trace(go.Scatter(x=df.index, y=df['ATR'],
                           line=dict(color='orange'), name='ATR'), row=3, col=1)
    
    fig.update_layout(height=1200, title_text="Enhanced Technical Analysis Dashboard")
    fig.show()

plot_technical_indicators(df)


## 4. Volatility Analysis

In [ ]:
def plot_volatility_analysis(df):
    fig = make_subplots(rows=2, cols=1, shared_xaxes=True,
                       subplot_titles=('Volatility Regimes', 'Intraday Volatility Pattern'))
    
    # Volatility regimes
    df['volatility_quantile'] = pd.qcut(df['volatility'], 4, labels=['Low', 'Moderate', 'High', 'Extreme'])
    colors = {'Low':'green', 'Moderate':'yellow', 'High':'orange', 'Extreme':'red'}
    fig.add_trace(go.Scatter(x=df.index, y=df['volatility'],
                           mode='markers', marker_color=df['volatility_quantile'].map(colors),
                           name='Volatility Regime'), row=1, col=1)
    
    # Intraday pattern
    hour_vol = df.groupby(df.index.hour)['volatility'].mean()
    fig.add_trace(go.Bar(x=hour_vol.index, y=hour_vol,
                       marker_color='purple', name='Hourly Volatility'), row=2, col=1)
    
    fig.update_layout(height=800, title_text="Advanced Volatility Analysis")
    fig.show()

plot_volatility_analysis(df)

## 5. Market Regime Analysis

In [ ]:
def detect_market_regime(df):
    df = df.copy()
    
    # Calculate volatility
    df['volatility'] = df['returns'].rolling(window=20).std()
    
    # Calculate trend strength using price changes
    df['trend'] = abs(df['close'].pct_change(20))
    
    # Define regime thresholds
    vol_threshold = df['volatility'].quantile(0.7)
    trend_threshold = df['trend'].quantile(0.7)
    
    # Classify market regimes
    conditions = [
        (df['trend'] >= trend_threshold) & (df['volatility'] >= vol_threshold),
        (df['trend'] >= trend_threshold) & (df['volatility'] < vol_threshold),
        (df['trend'] < trend_threshold) & (df['volatility'] >= vol_threshold),
        (df['trend'] < trend_threshold) & (df['volatility'] < vol_threshold)
    ]
    choices = ['STRONG_TREND', 'WEAK_TREND', 'VOLATILE_RANGE', 'QUIET_RANGE']
    df['regime'] = np.select(conditions, choices, default='QUIET_RANGE')
    
    return df

# Detect market regimes
df_regime = detect_market_regime(df)

# Plot market regimes
fig = make_subplots(rows=2, cols=1, shared_xaxes=True,
                    subplot_titles=('Price with Market Regimes', 'Regime Distribution'),
                    row_heights=[0.7, 0.3])

# Price plot with regime colors
colors = {'STRONG_TREND': 'red',
          'WEAK_TREND': 'orange',
          'VOLATILE_RANGE': 'purple',
          'QUIET_RANGE': 'blue'}

for regime in colors:
    mask = df_regime['regime'] == regime
    fig.add_trace(go.Scatter(x=df_regime[mask].index,
                             y=df_regime[mask]['close'],
                             name=regime,
                             mode='markers',
                             marker=dict(color=colors[regime], size=3)),
                  row=1, col=1)

# Regime distribution
regime_counts = df_regime['regime'].value_counts()
fig.add_trace(go.Bar(x=regime_counts.index,
                     y=regime_counts.values,
                     marker_color=list(map(lambda x: colors[x], regime_counts.index))),
              row=2, col=1)

fig.update_layout(height=800, title_text="Market Regime Analysis")
fig.show()

In [ ]:
def plot_market_regimes(df):
    # Create a copy of the dataframe and add required columns
    df_plot = df.copy()
    
    # Calculate trend using price changes (you can adjust the window)
    df_plot['trend'] = df_plot['close'].pct_change(20).abs()  # 20-period trend
    
    # Ensure we have all required columns
    required_columns = ['trend', 'volatility', 'returns', 'regime']
    if 'regime' not in df_plot.columns:
        # Calculate regime if not already present
        vol_threshold = df_plot['volatility'].quantile(0.7)
        trend_threshold = df_plot['trend'].quantile(0.7)
        
        conditions = [
            (df_plot['trend'] >= trend_threshold) & (df_plot['volatility'] >= vol_threshold),
            (df_plot['trend'] >= trend_threshold) & (df_plot['volatility'] < vol_threshold),
            (df_plot['trend'] < trend_threshold) & (df_plot['volatility'] >= vol_threshold),
            (df_plot['trend'] < trend_threshold) & (df_plot['volatility'] < vol_threshold)
        ]
        choices = ['STRONG_TREND', 'WEAK_TREND', 'VOLATILE_RANGE', 'QUIET_RANGE']
        df_plot['regime'] = np.select(conditions, choices, default='QUIET_RANGE')
    
    # Define colors for regimes
    colors = {
        'STRONG_TREND': 'red',
        'WEAK_TREND': 'orange',
        'VOLATILE_RANGE': 'purple',
        'QUIET_RANGE': 'blue'
    }
    
    # Create 3D visualization
    fig = px.scatter_3d(
        df_plot,
        x='trend',
        y='volatility',
        z='returns',
        color='regime',
        color_discrete_map=colors,
        title='3D Market Regime Analysis',
        labels={
            'trend': 'Trend Strength',
            'volatility': 'Volatility',
            'returns': 'Returns'
        }
    )
    
    # Enhance the 3D plot
    fig.update_layout(
        height=800,
        scene=dict(
            xaxis_title='Trend Strength',
            yaxis_title='Volatility',
            zaxis_title='Returns',
            camera=dict(
                up=dict(x=0, y=0, z=1),
                center=dict(x=0, y=0, z=0),
                eye=dict(x=1.5, y=1.5, z=1.5)
            )
        )
    )
    fig.show()
    
    # Regime duration analysis
    regime_changes = df_plot['regime'].ne(df_plot['regime'].shift()).cumsum()
    regime_durations = df_plot.groupby(regime_changes)['regime'].agg(['first', 'count'])
    
    # Create duration analysis plot
    plt.figure(figsize=(12, 6))
    sns.boxplot(x='first', y='count', data=regime_durations, palette=colors)
    plt.title('Market Regime Duration Analysis')
    plt.xlabel('Regime')
    plt.ylabel('Duration (15-min intervals)')
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()
    
    # Add regime distribution plot
    plt.figure(figsize=(10, 5))
    regime_dist = df_plot['regime'].value_counts()
    regime_dist.plot(kind='bar', color=[colors[r] for r in regime_dist.index])
    plt.title('Market Regime Distribution')
    plt.xlabel('Regime')
    plt.ylabel('Count')
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()
    
    # Print regime statistics
    print("\nRegime Statistics:")
    regime_stats = pd.DataFrame({
        'Count': df_plot['regime'].value_counts(),
        'Percentage': df_plot['regime'].value_counts(normalize=True) * 100,
        'Avg Duration': regime_durations.groupby('first')['count'].mean()
    })
    regime_stats['Percentage'] = regime_stats['Percentage'].round(2)
    regime_stats['Avg Duration'] = regime_stats['Avg Duration'].round(2)
    display(regime_stats)

# Call the function
plot_market_regimes(df)

## 6. Trading Signal Analysis

In [ ]:
def generate_signals(df):
    df = df.copy()
    
    # Calculate returns
    df['returns_1h'] = df['close'].shift(-4).pct_change(4)
    df['returns_4h'] = df['close'].shift(-16).pct_change(16)
    df['returns_24h'] = df['close'].shift(-96).pct_change(96)
    
    # Calculate volatility threshold
    volatility = df['close'].pct_change().rolling(window=20).std()
    vol_threshold = volatility.rolling(window=20).mean() * 2.0
    
    # Generate signals
    buy_signal = (
        (df['returns_1h'] > vol_threshold) &
        ((df['returns_4h'] > vol_threshold * 0.8) |
         (df['returns_24h'] > vol_threshold * 0.6))
    )
    
    sell_signal = (
        (df['returns_1h'] < -vol_threshold) &
        ((df['returns_4h'] < -vol_threshold * 0.8) |
         (df['returns_24h'] < -vol_threshold * 0.6))
    )
    
    df['signal'] = 0
    df.loc[buy_signal, 'signal'] = 1
    df.loc[sell_signal, 'signal'] = -1
    
    return df

# Generate signals
df_signals = generate_signals(df)

# Plot signals
fig = go.Figure()

# Plot price
fig.add_trace(go.Scatter(x=df_signals.index, y=df_signals['close'],
                         name='Price',
                         line=dict(color='gray', width=1)))

# Plot buy signals
buy_mask = df_signals['signal'] == 1
fig.add_trace(go.Scatter(x=df_signals[buy_mask].index,
                         y=df_signals[buy_mask]['close'],
                         name='Buy Signal',
                         mode='markers',
                         marker=dict(color='green', size=8, symbol='triangle-up')))

# Plot sell signals
sell_mask = df_signals['signal'] == -1
fig.add_trace(go.Scatter(x=df_signals[sell_mask].index,
                         y=df_signals[sell_mask]['close'],
                         name='Sell Signal',
                         mode='markers',
                         marker=dict(color='red', size=8, symbol='triangle-down')))

fig.update_layout(title='Trading Signals Analysis',
                  yaxis_title='Price',
                  height=600)

fig.show()

# Signal statistics
signal_stats = pd.DataFrame({
    'Buy Signals': [len(df_signals[df_signals['signal'] == 1])],
    'Sell Signals': [len(df_signals[df_signals['signal'] == -1])],
    'Signal Ratio': [len(df_signals[df_signals['signal'] != 0]) / len(df_signals)]
})

print("\nSignal Statistics:")
display(signal_stats)

## 7. Performance Metrics

In [ ]:
def calculate_performance_metrics(df):
    metrics = {}
    
    # Returns
    returns = df['close'].pct_change().dropna()
    metrics['Daily Return'] = returns.mean() * 24 * 4  # Convert to daily
    metrics['Daily Volatility'] = returns.std() * np.sqrt(24 * 4)
    metrics['Sharpe Ratio'] = metrics['Daily Return'] / metrics['Daily Volatility'] * np.sqrt(365)
    
    # Drawdown analysis
    cum_returns = (1 + returns).cumprod()
    rolling_max = cum_returns.expanding().max()
    drawdowns = (cum_returns - rolling_max) / rolling_max
    metrics['Max Drawdown'] = drawdowns.min()
    
    # Additional metrics
    metrics['Win Rate'] = len(returns[returns > 0]) / len(returns)
    metrics['Loss Rate'] = len(returns[returns < 0]) / len(returns)
    metrics['Profit Factor'] = abs(returns[returns > 0].sum() / returns[returns < 0].sum())
    
    return pd.Series(metrics)

# Calculate metrics
metrics = calculate_performance_metrics(df)

# Display metrics
print("Performance Metrics:")
display(pd.DataFrame(metrics).T)

# Plot cumulative returns
returns = df['close'].pct_change().dropna()
cum_returns = (1 + returns).cumprod()

fig = go.Figure()

fig.add_trace(go.Scatter(x=cum_returns.index, y=cum_returns,
                         name='Cumulative Returns',
                         line=dict(color='blue')))

fig.update_layout(title='Cumulative Returns',
                  yaxis_title='Returns',
                  height=500)

fig.show()

## 8. Correlation Analysis

In [ ]:
# Add volatility to indicators dataframe
df_indicators['volatility'] = df_indicators['close'].pct_change().rolling(window=20).std() * np.sqrt(365 * 24 * 4)  # Annualized

# Calculate correlations between technical indicators
correlation_columns = ['close', 'SMA_20', 'SMA_50', 'RSI', 'MACD', 'ATR', 'volatility']
correlation_matrix = df_indicators[correlation_columns].corr()

# Plot correlation heatmap
plt.figure(figsize=(10, 8))
sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', center=0, fmt='.2f')
plt.title('Correlation Matrix of Technical Indicators')
plt.xticks(rotation=45)
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()

# Plot rolling correlations
def plot_rolling_correlations(df, base_column='close', window=20):
    correlations = pd.DataFrame()
    for col in correlation_columns:
        if col != base_column:
            correlations[col] = df[base_column].rolling(window).corr(df[col])
    
    fig = go.Figure()
    
    for col in correlations.columns:
        fig.add_trace(go.Scatter(x=correlations.index,
                                y=correlations[col],
                                name=col))
    
    fig.update_layout(title=f'Rolling {window}-period Correlations with Price',
                      yaxis_title='Correlation',
                      height=500,
                      showlegend=True,
                      legend=dict(orientation='h', yanchor='bottom', y=1.02, xanchor='right', x=1))
    
    fig.show()

plot_rolling_correlations(df_indicators)

# Print strongest correlations
print("\nStrongest Correlations:")
correlations = []
for i in range(len(correlation_columns)):
    for j in range(i+1, len(correlation_columns)):
        corr = correlation_matrix.iloc[i,j]
        correlations.append({
            'Pair': f'{correlation_columns[i]} vs {correlation_columns[j]}',
            'Correlation': corr
        })

corr_df = pd.DataFrame(correlations)
corr_df = corr_df.sort_values('Correlation', key=abs, ascending=False)
display(corr_df.head())

## 9. Summary and Insights

Based on the above analysis, here are the key findings:

1. **Price Action**:
   - Observe the overall trend and major support/resistance levels
   - Note any significant price movements and their corresponding volumes

2. **Technical Indicators**:
   - Identify the effectiveness of different indicators
   - Note any significant divergences or confluences

3. **Market Regimes**:
   - Understand the distribution of different market regimes
   - Note how price behavior changes across regimes

4. **Trading Signals**:
   - Analyze the frequency and quality of signals
   - Note any patterns in signal generation

5. **Performance Metrics**:
   - Review key performance indicators
   - Identify areas for potential improvement

6. **Correlations**:
   - Understand relationships between different indicators
   - Note any strong correlations that could be exploited

These insights can be used to:
- Fine-tune trading parameters
- Adjust signal generation criteria
- Optimize risk management rules
- Improve overall trading strategy

In [4]:
# --------------------------------------------------
## Feature Engineering & Column Definition
# --------------------------------------------------

def create_features(df):
    """Create technical features and define feature columns"""
    # Calculate returns
    df = df.assign(
        returns_1h=df['close'].pct_change(4),
        returns_4h=df['close'].pct_change(16),
        returns_24h=df['close'].pct_change(96)
    )
    
    # Technical Indicators
    df['sma_20'] = df['close'].rolling(20).mean()
    df['sma_50'] = df['close'].rolling(50).mean()
    df['ema_20'] = df['close'].ewm(span=20).mean()
    df['ema_50'] = df['close'].ewm(span=50).mean()
    
    # RSI
    delta = df['close'].diff()
    gain = delta.where(delta > 0, 0)
    loss = -delta.where(delta < 0, 0)
    avg_gain = gain.rolling(14).mean()
    avg_loss = loss.rolling(14).mean()
    rs = avg_gain / avg_loss
    df['rsi'] = 100 - (100 / (1 + rs))
    
    # MACD
    exp12 = df['close'].ewm(span=12).mean()
    exp26 = df['close'].ewm(span=26).mean()
    df['macd'] = exp12 - exp26
    df['macd_signal'] = df['macd'].ewm(span=9).mean()
    
    # Bollinger Bands
    df['bb_upper'] = df['sma_20'] + 2*df['close'].rolling(20).std()
    df['bb_lower'] = df['sma_20'] - 2*df['close'].rolling(20).std()
    
    # Volume Features
    df['volume_ma_20'] = df['volume'].rolling(20).mean()
    df['volume_roc'] = df['volume'].pct_change(4)
    
    # Define feature columns
    feature_cols = [
        'returns_1h', 'returns_4h', 'returns_24h',
        'sma_20', 'sma_50', 'ema_20', 'ema_50',
        'rsi', 'macd', 'macd_signal',
        'bb_upper', 'bb_lower',
        'volume', 'volume_ma_20', 'volume_roc'
    ]
    
    # Add target column (next 4h return)
    df['target'] = (df['close'].shift(-16) > df['close']).astype(int)
    
    # Drop NA values after feature creation
    df = df.dropna()
    
    return df, feature_cols

# Apply feature engineering
df, feature_cols = create_features(df)

# Now proceed with the training code from previous answer
# This should resolve the 'feature_cols' not defined error

In [ ]:
# --------------------------------------------------
## Integrated Model Training & Visualization
# --------------------------------------------------

# New imports
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, classification_report
import plotly.figure_factory as ff
import plotly.express as px
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import shap
import matplotlib.pyplot as plt


DEVICE = 'mps' if torch.backends.mps.is_available() else 'cpu'

# Add this cell after your data loading and preprocessing
def prepare_training_data(df, test_size=0.2, lookback=24):
    """Prepare data for LSTM training with visualization"""
    # Create sequences
    X, y = [], []
    for i in range(len(df)-lookback-1):
        X.append(df[feature_cols].values[i:i+lookback])
        y.append(df['target'].values[i+lookback])
    
    X = np.array(X)
    y = np.array(y)
    
    # Train-test split
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=test_size, shuffle=False)
    
    # Visualize data distribution
    fig = px.pie(names=['Train', 'Test'], 
                 values=[len(X_train), len(X_test)],
                 title='Train-Test Split Distribution')
    fig.show()
    
    return X_train, X_test, y_train, y_test

# Add model architecture with visualization hooks
class TradingModel(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.lstm = nn.LSTM(input_dim, 64, batch_first=True, bidirectional=True)
        self.attention = nn.MultiheadAttention(128, 4, batch_first=True)
        self.fc = nn.Sequential(
            nn.Linear(128, 64),
            nn.SiLU(),
            nn.LayerNorm(64),
            nn.Linear(64, 1)
        )

    def forward(self, x):
        # x shape: [batch_size, sequence_length, input_dim]
        lstm_out, _ = self.lstm(x)
        # Store LSTM activations for later visualization
        self.lstm_activations = lstm_out
        
        # Pass LSTM outputs through the attention layer.
        attn_out, attn_weights = self.attention(lstm_out, lstm_out, lstm_out)
        # Store attention weights for visualization
        self.attention_weights = attn_weights
        
        # Use the last time step's output from the attention block for the final prediction.
        last_out = attn_out[:, -1, :]
        # Return a 2D tensor [batch_size, 1] without squeezing
        return self.fc(last_out)

def train_model_visual(X_train, y_train, X_val, y_val, feature_cols):
    """Interactive training with real-time visualization (debug logging included)"""
    # Initialize model, optimizer, and loss function
    model = TradingModel(len(feature_cols)).to(DEVICE)
    optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3)
    criterion = nn.BCEWithLogitsLoss()
    
    # Prepare DataLoader
    train_dataset = TensorDataset(
        torch.FloatTensor(X_train).to(DEVICE),
        torch.FloatTensor(y_train).to(DEVICE)
    )
    train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
    
    # Initialize metrics storage
    metrics = {
        'train_loss': [],
        'val_loss': [],
        'train_acc': [],
        'val_acc': []
    }
    
    # Training loop (each epoch creates a new figure to avoid duplicate legends)
    for epoch in range(10):
        # Create a new figure for this epoch
        fig = make_subplots(rows=2, cols=2,
                            specs=[[{'colspan': 2}, None], [{}, {}]],
                            subplot_titles=('Loss Progress', 'Attention Weights', 'LSTM Activations'))
        
        model.train()
        total_loss = 0
        correct = 0
        
        for batch_X, batch_y in train_loader:
            optimizer.zero_grad()
            outputs = model(batch_X)
            # Squeeze only for loss calculation
            outputs = outputs.squeeze(1)
            loss = criterion(outputs, batch_y)
            loss.backward()
            optimizer.step()
            
            total_loss += loss.item()
            preds = (torch.sigmoid(outputs) > 0.5).float()
            correct += (preds == batch_y).sum().item()
        
        # Evaluate on validation data
        model.eval()
        with torch.no_grad():
            val_outputs = model(torch.FloatTensor(X_val).to(DEVICE))
            val_outputs = val_outputs.squeeze(1)
            val_loss = criterion(val_outputs, torch.FloatTensor(y_val).to(DEVICE))
            val_preds = (torch.sigmoid(val_outputs) > 0.5).float()
            val_acc = (val_preds.cpu() == torch.FloatTensor(y_val)).sum().item() / len(y_val)
        
        # Update metrics
        metrics['train_loss'].append(total_loss / len(train_loader))
        metrics['val_loss'].append(val_loss.item())
        metrics['train_acc'].append(correct / len(X_train))
        metrics['val_acc'].append(val_acc)
        
        # --- Debug Logging ---
        print(f"Epoch {epoch+1}:")
        print("  LSTM activations shape:", model.lstm_activations.shape)
        print("  Attention weights shape:", model.attention_weights.shape)
        
        # --- Plot Loss Metrics ---
        # (Here we add the legend only on the first epoch)
        fig.add_trace(go.Scatter(
            x=list(range(epoch+1)),
            y=metrics['train_loss'],
            name='Train Loss',
            mode='lines+markers',
            showlegend=(epoch == 0)
        ), row=1, col=1)
        fig.add_trace(go.Scatter(
            x=list(range(epoch+1)),
            y=metrics['val_loss'],
            name='Val Loss',
            mode='lines+markers',
            showlegend=(epoch == 0)
        ), row=1, col=1)
        
        # --- Plot Attention Weights ---
        attn_weights = model.attention_weights
        if attn_weights is not None and attn_weights.numel() > 0:
            # Take attention weights from the last sample and first head
            # (attn_weights shape: [batch_size, num_heads, seq_len, seq_len])
            attn_last = attn_weights[-1].detach().cpu().numpy()  # shape: (num_heads, seq_len, seq_len)
            fig.add_trace(go.Heatmap(
                z=attn_last[0],
                colorscale='Viridis',
                showscale=False,
                name="Attention Weights"
            ), row=2, col=1)
        else:
            print("Warning: Attention weights are empty!")
        
        # --- Plot LSTM Activations ---
        lstm_acts = model.lstm_activations
        # Use the last time step from the last sample (resulting in a 1D array, which we expand to 2D)
        lstm_acts_last = lstm_acts[-1].detach().cpu().numpy()  # shape: (feature_dim,)
        lstm_acts_last_2d = np.expand_dims(lstm_acts_last, axis=0)  # shape becomes (1, feature_dim)
        fig.add_trace(go.Heatmap(
            z=lstm_acts_last_2d,
            colorscale='Plasma',
            showscale=False,
            name="LSTM Activations"
        ), row=2, col=2)
        
        fig.update_layout(
            height=800,
            title=f"Training Progress - Epoch {epoch+1}",
            showlegend=True
        )
        fig.show()
        
    return model
try:
    # Prepare data
    X_train, X_test, y_train, y_test = prepare_training_data(df)
    
    # Train with visualization
    model = train_model_visual(X_train, y_train, X_test, y_test, feature_cols)
    
    # Final evaluation
    with torch.no_grad():
        test_outputs = model(torch.FloatTensor(X_test).to(DEVICE)).squeeze()
        test_probs = torch.sigmoid(test_outputs).cpu().numpy()
        test_preds = (test_probs > 0.5).astype(int)
    
    # Confusion matrix
    cm = confusion_matrix(y_test, test_preds)
    fig = ff.create_annotated_heatmap(
        z=cm,
        x=['Sell', 'Buy'],
        y=['Sell', 'Buy'],
        colorscale='Blues'
    )
    fig.update_layout(title='Confusion Matrix')
    fig.show()
    
    # Classification report
    print(classification_report(y_test, test_preds, target_names=['Sell', 'Buy']))
    # Feature importance using SHAP
    try:
        # Log the shape of X_test used for SHAP explanation
        background = torch.FloatTensor(X_train[:100]).to(DEVICE)
        test_data = torch.FloatTensor(X_test[:100]).to(DEVICE)
        print("Background data shape:", background.shape)  # For example, (100, lookback, n_features)
        print("Test data shape:", test_data.shape)
        
        explainer = shap.DeepExplainer(model, background)
        raw_shap_values = explainer.shap_values(test_data)  # raw_shap_values is a list
        # Log raw SHAP values shape
        print("Raw shap_values[0] shape:", raw_shap_values[0].shape)
        
        # For sequence models, raw_shap_values[0] has shape (samples, lookback, n_features),
        # but shap.summary_plot expects a 2D array. We take the average over the time dimension.
        shap_values_agg = np.mean(raw_shap_values[0], axis=1)  # now shape is (samples, n_features)
        X_test_agg = np.mean(X_test[:100], axis=1)             # aggregated to (samples, n_features)
        
        fig, ax = plt.subplots(figsize=(12, 6))
        shap.summary_plot(shap_values_agg, X_test_agg, feature_names=feature_cols, plot_type='bar')
        plt.title('SHAP Feature Importance')
        plt.show()
    except Exception as e:
        print(f"SHAP analysis failed: {str(e)}")

    # fig, ax = plt.subplots(figsize=(12, 6))
    # shap.summary_plot(shap_values, X_test[:100], feature_names=feature_cols, plot_type='bar')
    # plt.title('SHAP Feature Importance')
    # plt.show()
    
    
except Exception as e:
    print(f"Training failed: {str(e)}")

In [ ]:
# Feature correlation matrix
corr_matrix = df[feature_cols].corr()
plt.figure(figsize=(16, 12))
sns.heatmap(corr_matrix, annot=False, cmap='coolwarm', vmin=-1, vmax=1)
plt.title('Feature Correlation Matrix')
plt.show()